# Blood-cell classification (BloodMNIST): live demo

Kancharla Aditya Eswar · Dual Degree Project · Guide: Prof. R. Jayaganthan · IIT Madras

This notebook re-runs the key results of the project from the public code
(https://github.com/Aditya-Eswar12/bloodmnist-ddp):

1. **Data**: BloodMNIST, 17,092 single-cell images, 8 classes, 64×64 pixels.
2. **A careful baseline**: train the small CNN with validation-based checkpoint selection; score the test set once.
3. **Stain-physics segmentation**: nucleus and cytoplasm without labels, scored against expert masks.
4. **External validation**: other laboratories' images; scale matching and colour normalisation.
5. **Calibration**: is the model's confidence trustworthy?
6. **The web service**: one image in, a prediction out; a slide-level report.
7. **All saved results** from the full experiments (3 seeds, masked pre-training, GAN).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all (about 6–8 minutes).

## 0 · Setup: get the code and the libraries

In [ ]:
import os, sys, subprocess, json, time
IN_COLAB = "google.colab" in sys.modules
EPOCHS = int(os.environ.get("DEMO_EPOCHS", "15"))          # 15 epochs = the reported setting

def sh(cmd):
    """Run a shell command and print its output line by line."""
    print(">>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("command failed: " + cmd)

REPO = "/content/bloodmnist-ddp" if IN_COLAB else os.environ.get("REPO", os.path.abspath("bloodmnist-ddp"))
if not os.path.exists(REPO):
    sh(f"git clone -q https://github.com/Aditya-Eswar12/bloodmnist-ddp {REPO}")
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if IN_COLAB:
    sh("pip -q install medmnist fastapi python-multipart httpx")

import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## 1 · The data: BloodMNIST

In [ ]:
import numpy as np
from medmnist import BloodMNIST
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/bloodmnist_64.npz"):
    BloodMNIST(split="train", download=True, size=64, root="data")   # downloads all three splits (~150 MB)
d = np.load("data/bloodmnist_64.npz")
for s in ("train", "val", "test"):
    print(f"{s:5s}", d[f"{s}_images"].shape, d[f"{s}_labels"].shape)

from data import CLASSES
counts = np.bincount(d["train_labels"].ravel(), minlength=8)
for c, n in zip(CLASSES, counts):
    print(f"{c:22s} {n:5d}  class weight {counts.sum() / (8 * n):.2f}")

In [ ]:
import matplotlib.pyplot as plt
x, y = d["test_images"], d["test_labels"].ravel()
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for c in range(8):
    idx = np.where(y == c)[0][:2]
    for r in range(2):
        ax[r, c].imshow(x[idx[r]]); ax[r, c].axis("off")
    ax[0, c].set_title(CLASSES[c].replace("_", " "), fontsize=9)
plt.suptitle("BloodMNIST test images, two per class"); plt.show()

## 2 · A careful baseline: the small CNN

* class-weighted loss + MixUp (as in the July model);
* the checkpoint with the best **validation** macro-F1 is kept;
* the **test** set is scored once, at the end.

Reported over 3 seeds: **97.9 ± 0.1 %** accuracy, macro-F1 0.979. This run trains one seed.

In [ ]:
t0 = time.time()
sh(f"python -u src/train_baseline.py --model cnn --epochs {EPOCHS} --seeds 0 --out results_live")
print(f"training took {time.time() - t0:.0f} s")
r = json.load(open("results_live/baseline_cnn_s0.json"))
t = r["test"]
print(f"\nTEST accuracy {t['accuracy']:.4f} | macro-F1 {t['macro_f1']:.4f} | monocyte recall {t['monocyte_recall']:.3f}")

In [ ]:
cm = np.array(t["confusion"])
short = ["baso", "eos", "ebl", "IG", "lymph", "mono", "neut", "plt"]
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues")
for i in range(8):
    for j in range(8):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if i == j else "black")
ax.set_xticks(range(8)); ax.set_xticklabels(short); ax.set_yticks(range(8)); ax.set_yticklabels(short)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Test confusion matrix (counts)")
plt.show()
for c, rec, pre in zip(CLASSES, t["recall"], t["precision"]):
    print(f"{c:22s} recall {rec:.3f}  precision {pre:.3f}")

## 3 · Stain physics: nucleus and cytoplasm without labels

Optical density OD = −log(I / I₀) (Beer–Lambert law) is proportional to the amount of dye. The nucleus has the
highest **red-channel** OD; a 3-level Otsu threshold separates it. Cytoplasm = stained pixels that absorb more red
than blue light, attached to the nucleus.

In [ ]:
from compartments import segment
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for k, c in enumerate([5, 3, 6, 4, 1, 0]):
    img = x[np.where(y == c)[0][3]]
    ax[0, k].imshow(img); ax[0, k].set_title(CLASSES[c].replace("_", " "), fontsize=9)
    ax[1, k].imshow(segment(img), cmap="Purples", vmin=0, vmax=2)
    for a in ax[:, k]: a.axis("off")
ax[1, 0].set_ylabel("stain-derived map")
plt.suptitle("Top: image · Bottom: background (white), cytoplasm (light), nucleus (dark)"); plt.show()

Scored against **expert masks** (Zheng et al., 2018). The method was designed on images 1–50 of Dataset 2 (and Dataset 1); images 51–100 are the held-out test.

In [ ]:
if not os.path.exists("data/segmentation_WBC"):
    sh("git clone -q https://github.com/zxaoyou/segmentation_WBC data/segmentation_WBC")
sh("python src/eval_masks.py --test")

## 4 · External validation: other laboratories' images

* Dataset 1: 300 images, a different stain. Dataset 2: 100 CellaVision images from other laboratories.
* **scale**: crop around the cell so it fills 41 % of the frame, as in BloodMNIST.
* **stain**: Reinhard colour normalisation to BloodMNIST's average colours.

In [ ]:
from external_eval import load_external, predict_supervised, VARIANTS
ckpt = "checkpoints/baseline_cnn_s0.pt"
print(f"{'variant':12s} {'Dataset 1 (other stain)':>24s} {'Dataset 2 (other labs)':>24s}")
for v in VARIANTS:
    accs = []
    for ds in ("Dataset 1", "Dataset 2"):
        xe, ye = load_external(ds, v)
        accs.append((predict_supervised(ckpt, xe) == ye).mean())
    print(f"{v:12s} {accs[0]:24.3f} {accs[1]:24.3f}")
print("\nHome test accuracy:", round(t["accuracy"], 3))

In [ ]:
# why: the cell fills less of the frame in the external images
from eval_masks import load as load_expert          # expert masks that come with the external dataset
ours = np.median([(segment(im) > 0).mean() for im in x[:500]])
expert = np.median([(load_expert("Dataset 2", i)[1] > 0).mean() for i in range(1, 101)])
print(f"median share of the 64x64 frame covered by the cell")
print(f"  BloodMNIST (my stain maps):            {ours:.2f}")
print(f"  external CellaVision (expert masks):   {expert:.2f}")

## 5 · Calibration: does 90 % confidence mean 90 % correct?

In [ ]:
import torch.nn.functional as F
from calibration import logits_of, ece, fit_temperature
from data import load_split
from train_baseline import build
blob = torch.load(ckpt, map_location="cpu")
model = build(blob["model"]); model.load_state_dict(blob["state_dict"])
(xv, yv), (xt, yt) = load_split("val"), load_split("test")
lv, lt = logits_of(model, xv), logits_of(model, xt)
T = fit_temperature(lv, yv)
print(f"expected calibration error: before {ece(lt.softmax(1), yt)[0]:.3%} | after temperature scaling (T = {T:.2f}) "
      f"{ece((lt / T).softmax(1), yt)[0]:.3%}")

## 6 · The web service (FastAPI): an image in, a prediction out

In [ ]:
import io
os.environ["SERVE_NO_AUTOAPP"] = "1"
sys.path.insert(0, os.path.join(REPO, "serve"))
from fastapi.testclient import TestClient
from PIL import Image
from app import create_app
client = TestClient(create_app(db_path="demo.sqlite", ckpt=ckpt))

def png(arr):
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format="PNG"); return buf.getvalue()

i = int(np.where(y == 5)[0][0])                                   # a monocyte
resp = client.post("/predict", files={"file": ("cell.png", png(x[i]), "image/png")}).json()
print("true class:", CLASSES[y[i]], "| predicted:", resp["class"], "| confidence:", resp["confidence"])

# a simulated normal smear: 200 white cells in normal proportions + 20 platelets, drawn from test images
from demo_cohort import build_slide, PROFILES
xs, truth = build_slide(np.random.default_rng(0), torch.from_numpy(x).permute(0, 3, 1, 2), y, PROFILES["normal"])
files = [("files", (f"c{k}.png", png(xs[k].permute(1, 2, 0).numpy()), "image/png")) for k in range(len(xs))]
client.post("/slides/NORMAL-1/cells", files=files)
print(client.post("/slides/NORMAL-1/report").json()["text"])

## 7 · All saved results (full experiments: 3 seeds each)

These come from the complete runs on the laptop (masked pre-training, label efficiency, GAN, external tests), stored
in `results/`. Regenerate everything with `bash run_all.sh` and `bash run_phase2.sh`.

In [ ]:
try:
    from IPython.display import Markdown, Image as Img, display
except ImportError:                       # outside Jupyter
    Markdown = Img = None
text = open("results/RESULTS.md", encoding="utf-8").read()
if Markdown:
    display(Markdown(text))
    for f in ["fig4_label_efficiency.png", "fig5_external.png", "fig6_gan_before_after.png"]:
        if os.path.exists("figures/" + f):
            display(Img("figures/" + f, width=750))
else:
    print(text[:1500])